# VAE Training Notebook — HPC-Ready

Complete, resumable training workflow for `ConvVAE3D` / `UNetVAE3D`.  
Runs on an HPC compute node where the Jupyter kernel has direct GPU access.

**Prerequisites:**
- Clone this repo on the HPC node and install: `pip install -e .`
- Copy your processed dataset (Zarr + Parquet + splits.json) to the HPC filesystem
- Set `POREGEN_DATA_ROOT` and `POREGEN_RUN_DIR` env vars (or edit the config cell)

---
## A) HPC + Environment Check

In [ ]:
import sys, os, platform

print(f"Python   : {sys.version}")
print(f"Platform : {platform.node()} ({platform.system()} {platform.release()})")

# ── PyTorch + CUDA ──
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"cuDNN    : {torch.backends.cudnn.version()}")
print(f"CUDA available : {torch.cuda.is_available()}")

assert torch.cuda.is_available(), (
    "No CUDA device found! Make sure you're running on a compute node with GPUs. "
    "On SLURM: srun --gres=gpu:1 --pty jupyter-lab"
)

N_GPUS = torch.cuda.device_count()
print(f"GPUs ({N_GPUS}):")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {props.name}  ({props.total_mem / 1e9:.1f} GB, sm_{props.major}{props.minor})")

# ── poregen package ──
# The repo must be cloned on the HPC and installed with: pip install -e .
# This makes all poregen.* imports available to the Jupyter kernel.
try:
    import poregen
    print(f"\nporegen  : imported OK")
except ImportError:
    raise ImportError(
        "Cannot import `poregen`. On your HPC node, run:\n"
        "  cd /path/to/GenAI && pip install -e .\n"
        "Then restart the Jupyter kernel."
    )

---
## B) Configuration

Single config dict — edit this cell for your experiment.  
Paths use environment variables with sensible fallbacks.

In [ ]:
CFG = dict(
    # ── Paths ──────────────────────────────────────────────────────────
    # On HPC: export POREGEN_DATA_ROOT=/localscratch/$USER/data/processed
    # On HPC: export POREGEN_RUN_DIR=$SCRATCH/runs/vae
    DATA_ROOT   = os.environ.get("POREGEN_DATA_ROOT", "../data/processed"),
    RUN_DIR     = os.environ.get("POREGEN_RUN_DIR", "../runs/vae"),
    EXP_NAME    = "conv_baseline",  # subdirectory inside RUN_DIR

    # ── Model ──────────────────────────────────────────────────────────
    model_name      = "conv",        # "conv" (ConvVAE3D) or "unet" (UNetVAE3D)
    z_channels      = 8,
    base_channels   = 32,
    n_blocks        = 2,
    patch_size      = 64,
    use_compile     = False,          # torch.compile (PyTorch 2.x, experimental)

    # ── Data ───────────────────────────────────────────────────────────
    # patch_size and stride must match what build_dataset produced
    batch_size      = 4,
    num_workers     = 4,
    pin_memory      = True,
    prefetch_factor = 2,

    # ── Training ───────────────────────────────────────────────────────
    lr              = 1e-4,
    weight_decay    = 1e-5,
    epochs          = 5,              # number of step-based epochs
    steps_per_epoch = 500,             # optimizer steps per epoch
    val_batches     = 20,              # validation batches per eval
    eval_every      = 100,             # eval every N training steps
    save_every      = 500,             # checkpoint every N steps

    # ── AMP + Grad clipping ────────────────────────────────────────────
    use_amp         = True,
    max_grad_norm   = 1.0,             # None to disable clipping

    # ── Scheduler ──────────────────────────────────────────────────────
    scheduler_type  = "cosine",        # "cosine", "onecycle", or None

    # ── KL / beta schedule ─────────────────────────────────────────────
    kl_warmup_steps = 2000,
    kl_max_beta     = 1.0,
    kl_free_bits    = 0.25,

    # ── Loss weights ───────────────────────────────────────────────────
    xct_loss_type   = "l1",            # "l1", "mse", "charbonnier"
    xct_weight      = 1.0,
    mask_bce_weight = 1.0,
    mask_dice_weight= 1.0,
    use_tversky     = False,
    tversky_alpha   = 0.3,
    tversky_beta    = 0.7,
    mask_pos_weight = None,            # float or None

    # ── Seed ───────────────────────────────────────────────────────────
    seed            = 42,

    # ── DDP (multi-GPU) ────────────────────────────────────────────────
    use_ddp         = False,           # enable multi-GPU via mp.spawn

    # ── Overfit test ───────────────────────────────────────────────────
    overfit_one_batch = False,         # set True to verify model can memorize

    # ── Logging ────────────────────────────────────────────────────────
    log_images_every = 200,            # log reconstruction slices every N steps
)

# Derived
CFG["total_steps"] = CFG["epochs"] * CFG["steps_per_epoch"]
print(f"Total training steps: {CFG['total_steps']:,}")
print(f"Model: {CFG['model_name']} | Patch: {CFG['patch_size']} | BS: {CFG['batch_size']}")

---
## C) Dataset + DataLoaders

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from poregen.dataset.loader import PatchDataset
from poregen.training import seed_everything

seed_everything(CFG["seed"])

DATA_ROOT = Path(CFG["DATA_ROOT"])
# Dataset must exist on the HPC filesystem.
# Typically: copy your processed/ folder to $SCRATCH or /localscratch.
assert (DATA_ROOT / "patch_index.parquet").exists(), (
    f"patch_index.parquet not found in {DATA_ROOT}. "
    f"Copy your processed dataset to the HPC or set POREGEN_DATA_ROOT."
)

train_ds = PatchDataset(DATA_ROOT / "patch_index.parquet", DATA_ROOT, split="train")
val_ds   = PatchDataset(DATA_ROOT / "patch_index.parquet", DATA_ROOT, split="val")

print(f"Train patches: {len(train_ds):,}  |  Val patches: {len(val_ds):,}")

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    prefetch_factor=CFG["prefetch_factor"],
    drop_last=True,
    persistent_workers=CFG["num_workers"] > 0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    prefetch_factor=CFG["prefetch_factor"],
    persistent_workers=CFG["num_workers"] > 0,
)

# ── Verify a batch ──
sample = next(iter(train_loader))
print(f"xct  shape: {sample['xct'].shape}  dtype: {sample['xct'].dtype}  "
      f"range: [{sample['xct'].min():.3f}, {sample['xct'].max():.3f}]")
print(f"mask shape: {sample['mask'].shape}  dtype: {sample['mask'].dtype}  "
      f"range: [{sample['mask'].min():.3f}, {sample['mask'].max():.3f}]")
assert sample['xct'].shape == (CFG['batch_size'], 1, CFG['patch_size'], CFG['patch_size'], CFG['patch_size']), \
    f"Unexpected shape: {sample['xct'].shape}"
print(f"Mean porosity: {sample['porosity'].mean():.4f}")

---
## D) Model Creation

In [ ]:
from poregen.models.vae import build_vae
from poregen.training import select_device, get_autocast_dtype, make_scaler

device = select_device()  # GPU 0 by default
autocast_dtype = get_autocast_dtype(device)
print(f"Device: {device}  |  AMP dtype: {autocast_dtype}")

model = build_vae(
    CFG["model_name"],
    z_channels=CFG["z_channels"],
    base_channels=CFG["base_channels"],
    n_blocks=CFG["n_blocks"],
    patch_size=CFG["patch_size"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {type(model).__name__}")
print(f"Parameters: {n_params:,} total  |  {n_train:,} trainable")
print(f"Latent shape: (B, {CFG['z_channels']}, {model.cfg.latent_spatial}, {model.cfg.latent_spatial}, {model.cfg.latent_spatial})")

# Optional: torch.compile for PyTorch 2.x
if CFG["use_compile"] and hasattr(torch, "compile"):
    model = torch.compile(model)
    print("Model compiled with torch.compile")

---
## E) Loss + Metrics Setup

In [ ]:
from poregen.losses import compute_total_loss
from poregen.metrics import segmentation_metrics, latent_stats, mae, psnr

# Build the loss config dict from CFG
loss_cfg = {
    "xct_loss_type":    CFG["xct_loss_type"],
    "xct_weight":       CFG["xct_weight"],
    "mask_bce_weight":  CFG["mask_bce_weight"],
    "mask_dice_weight": CFG["mask_dice_weight"],
    "use_tversky":      CFG["use_tversky"],
    "tversky_alpha":    CFG["tversky_alpha"],
    "tversky_beta":     CFG["tversky_beta"],
    "kl_free_bits":     CFG["kl_free_bits"],
    "kl_warmup_steps":  CFG["kl_warmup_steps"],
    "kl_max_beta":      CFG["kl_max_beta"],
}

def loss_fn(output, batch, step):
    return compute_total_loss(output, batch, step, loss_cfg)

print("Loss config:")
for k, v in loss_cfg.items():
    print(f"  {k}: {v}")

---
## F) Optimizer + Scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

scaler = make_scaler(device)

# ── LR scheduler ──
scheduler = None
if CFG["scheduler_type"] == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["total_steps"], eta_min=CFG["lr"] * 0.01,
    )
    print(f"Scheduler: CosineAnnealingLR (T_max={CFG['total_steps']})")
elif CFG["scheduler_type"] == "onecycle":
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG["lr"] * 10,
        total_steps=CFG["total_steps"],
        pct_start=0.1,
    )
    print(f"Scheduler: OneCycleLR (max_lr={CFG['lr'] * 10})")
else:
    print("Scheduler: None (constant LR)")

print(f"Optimizer: AdamW (lr={CFG['lr']}, wd={CFG['weight_decay']})")

---
## G) Resumable Training — Checkpoint Setup

Auto-resumes from `last.ckpt` if it exists in the run directory.

In [ ]:
from poregen.training import save_checkpoint, load_checkpoint
from torch.utils.tensorboard import SummaryWriter

RUN_DIR = Path(CFG["RUN_DIR"]) / CFG["EXP_NAME"]
CKPT_DIR = RUN_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# TensorBoard writer
tb_writer = SummaryWriter(log_dir=str(RUN_DIR / "tb_logs"))
print(f"TensorBoard logs: {RUN_DIR / 'tb_logs'}")
print(f"  → Launch: tensorboard --logdir {RUN_DIR / 'tb_logs'}")

# ── Auto-resume ──
start_step = 0
best_val_loss = float("inf")
last_ckpt = CKPT_DIR / "last.pt"

if last_ckpt.exists():
    start_step, meta = load_checkpoint(
        last_ckpt, model, optimizer, scaler,
        scheduler=scheduler,
        map_location=device,
        restore_rng=True,
    )
    best_val_loss = meta.get("best_val_loss", float("inf"))
    print(f"Resumed from step {start_step:,} (best val loss: {best_val_loss:.4f})")
else:
    print(f"No checkpoint found — starting fresh from step 0")

print(f"Run directory: {RUN_DIR}")

---
## H) Training Loop (Step-Based Epochs)

Each epoch = `steps_per_epoch` optimizer updates.  
DataLoader cycles infinitely — no dependency on dataset size for timing.

In [ ]:
import time
import matplotlib.pyplot as plt
from poregen.training.engine import train_step, eval_step

def _infinite(loader):
    """Cycle through a DataLoader forever."""
    while True:
        yield from loader


def log_reconstruction_images(writer, output, batch, step, prefix="val"):
    """Log mid-slice XCT and mask reconstructions to TensorBoard."""
    with torch.no_grad():
        xct_pred = torch.sigmoid(output.xct_logits[0, 0]).float()    # (D, H, W)
        mask_pred = (torch.sigmoid(output.mask_logits[0, 0]) > 0.5).float()
        xct_gt = batch["xct"][0, 0].float().to(xct_pred.device)
        mask_gt = batch["mask"][0, 0].float().to(mask_pred.device)

        mid = xct_pred.shape[0] // 2
        # Stack GT and pred side-by-side for each channel
        xct_img = torch.cat([xct_gt[mid], xct_pred[mid]], dim=1).unsqueeze(0)   # (1,H,2W)
        mask_img = torch.cat([mask_gt[mid], mask_pred[mid]], dim=1).unsqueeze(0)

        writer.add_image(f"{prefix}/xct_gt_vs_pred", xct_img, step)
        writer.add_image(f"{prefix}/mask_gt_vs_pred", mask_img, step)


# ── Overfit-one-batch mode ──
if CFG["overfit_one_batch"]:
    print("⚠ OVERFIT MODE: training on a single batch to verify model can memorize")
    fixed_batch = next(iter(train_loader))

# ── Main loop ──
train_iter = _infinite(train_loader)
val_iter = _infinite(val_loader)
history = []
global_step = start_step

for epoch in range(CFG["epochs"]):
    epoch_t0 = time.time()
    epoch_losses = []

    for local_step in range(CFG["steps_per_epoch"]):
        if global_step >= CFG["total_steps"]:
            break

        # Get batch
        batch = fixed_batch if CFG["overfit_one_batch"] else next(train_iter)

        # Training step
        losses = train_step(
            model, batch, optimizer, scaler, loss_fn,
            step=global_step, device=device, autocast_dtype=autocast_dtype,
            max_grad_norm=CFG["max_grad_norm"],
            scheduler=scheduler,
        )

        # Log to TensorBoard
        for k, v in losses.items():
            tb_writer.add_scalar(f"train/{k}", v, global_step)
        if scheduler is not None:
            tb_writer.add_scalar("train/lr", scheduler.get_last_lr()[0], global_step)

        epoch_losses.append(losses["total"])
        history.append({"step": global_step, "split": "train", **losses})

        # ── Periodic validation ──
        if (global_step + 1) % CFG["eval_every"] == 0:
            model.eval()
            val_loss_accum = []
            last_val_output = None
            last_val_batch = None
            for _ in range(CFG["val_batches"]):
                vb = next(val_iter)
                vl, vo = eval_step(model, vb, loss_fn, global_step, device, autocast_dtype)
                val_loss_accum.append(vl["total"])
                last_val_output = vo
                last_val_batch = vb

            avg_val_loss = sum(val_loss_accum) / len(val_loss_accum)
            tb_writer.add_scalar("val/total", avg_val_loss, global_step)

            # Compute metrics on last val batch
            if last_val_output is not None:
                seg_m = segmentation_metrics(
                    last_val_output.mask_logits.float(),
                    last_val_batch["mask"].to(device).float(),
                )
                for k, v in seg_m.items():
                    tb_writer.add_scalar(f"val/{k}", v, global_step)

                xct_mae = mae(last_val_output.xct_logits.float(),
                              last_val_batch["xct"].to(device).float())
                xct_psnr = psnr(last_val_output.xct_logits.float(),
                               last_val_batch["xct"].to(device).float())
                tb_writer.add_scalar("val/xct_mae", xct_mae.item(), global_step)
                tb_writer.add_scalar("val/xct_psnr", xct_psnr.item(), global_step)

                lat = latent_stats(last_val_output.mu.float(), last_val_output.logvar.float())
                for k, v in lat.items():
                    tb_writer.add_scalar(f"val/latent_{k}", v, global_step)

            # Log reconstruction images
            if (global_step + 1) % CFG["log_images_every"] == 0 and last_val_output is not None:
                log_reconstruction_images(tb_writer, last_val_output, last_val_batch, global_step)

            # Save best
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                save_checkpoint(
                    CKPT_DIR / "best.pt", model, optimizer, scaler,
                    step=global_step + 1, scheduler=scheduler,
                    metadata={"best_val_loss": best_val_loss},
                )

            print(f"  [step {global_step + 1:>6}] val_loss={avg_val_loss:.4f} "
                  f"dice={seg_m.get('dice_all', 0):.3f} "
                  f"psnr={xct_psnr.item():.1f}dB "
                  f"{'★ best' if avg_val_loss <= best_val_loss else ''}")
            model.train()

        # ── Periodic checkpoint ──
        if (global_step + 1) % CFG["save_every"] == 0:
            save_checkpoint(
                CKPT_DIR / "last.pt", model, optimizer, scaler,
                step=global_step + 1, scheduler=scheduler,
                metadata={"best_val_loss": best_val_loss},
            )

        global_step += 1

    # Epoch summary
    epoch_time = time.time() - epoch_t0
    avg_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else 0
    steps_sec = len(epoch_losses) / epoch_time if epoch_time > 0 else 0
    print(f"Epoch {epoch + 1}/{CFG['epochs']} | "
          f"avg_loss={avg_loss:.4f} | {steps_sec:.1f} steps/s | "
          f"{epoch_time:.0f}s")

# Final checkpoint
save_checkpoint(
    CKPT_DIR / "last.pt", model, optimizer, scaler,
    step=global_step, scheduler=scheduler,
    metadata={"best_val_loss": best_val_loss},
)
tb_writer.close()
print(f"\nDone! {global_step:,} steps completed. Best val loss: {best_val_loss:.4f}")

---
## I) Optional: Multi-GPU (DDP)

**Recommended approach for notebooks:** convert to script and use `torchrun`.

```bash
# 1) Convert this notebook to a Python script
jupyter nbconvert --to script notebooks/train_vae.ipynb --output train_vae_script

# 2) Launch with torchrun (2 GPUs)
torchrun --nproc_per_node=2 notebooks/train_vae_script.py
```

If you want to stay in-notebook, the cell below uses `mp.spawn` to wrap the
model in DDP. **This is experimental** — `torchrun` is more reliable.

In [ ]:
# ── DDP from notebook (experimental) ──
# Only run this cell if you set CFG["use_ddp"] = True
# and want multi-GPU without leaving the notebook.

import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler


def ddp_train_worker(rank, world_size, cfg):
    """Single DDP worker — one per GPU."""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    # Rebuild everything per-worker
    seed_everything(cfg["seed"] + rank)
    model = build_vae(
        cfg["model_name"],
        z_channels=cfg["z_channels"],
        base_channels=cfg["base_channels"],
        n_blocks=cfg["n_blocks"],
        patch_size=cfg["patch_size"],
    ).to(device)
    model = DDP(model, device_ids=[rank])

    data_root = Path(cfg["DATA_ROOT"])
    train_ds = PatchDataset(data_root / "patch_index.parquet", data_root, split="train")
    sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"],
        sampler=sampler, num_workers=cfg["num_workers"],
        pin_memory=True, drop_last=True,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scaler = make_scaler(device)
    autocast_dt = get_autocast_dtype(device)

    # Simple training loop (abridged — reuses train_step)
    train_it = iter(train_loader)
    for step in range(cfg["total_steps"]):
        try:
            batch = next(train_it)
        except StopIteration:
            sampler.set_epoch(step)
            train_it = iter(train_loader)
            batch = next(train_it)

        losses = train_step(
            model, batch, optimizer, scaler, loss_fn,
            step=step, device=device, autocast_dtype=autocast_dt,
            max_grad_norm=cfg["max_grad_norm"],
        )

        if rank == 0 and (step + 1) % 100 == 0:
            print(f"[DDP step {step + 1}] loss={losses['total']:.4f}")

    if rank == 0:
        save_checkpoint(
            Path(cfg["RUN_DIR"]) / cfg["EXP_NAME"] / "checkpoints" / "last_ddp.pt",
            model.module, optimizer, scaler, step=cfg["total_steps"],
        )
    dist.destroy_process_group()


if CFG["use_ddp"] and N_GPUS >= 2:
    print(f"Launching DDP training on {N_GPUS} GPUs...")
    # mp.spawn forks — make sure this cell is the only thing running
    mp.spawn(ddp_train_worker, args=(N_GPUS, CFG), nprocs=N_GPUS, join=True)
    print("DDP training complete.")
elif CFG["use_ddp"]:
    print(f"DDP requested but only {N_GPUS} GPU(s) found. Use single-GPU mode (section H).")
else:
    print("DDP disabled — using single-GPU training (section H).")

---
## J) Evaluation / Sanity Checks

### J.1) Quick Loss Plot

In [ ]:
import matplotlib.pyplot as plt

train_h = [r for r in history if r["split"] == "train"]
if train_h:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

    axes[0].plot([r["step"] for r in train_h], [r["total"] for r in train_h], lw=0.6)
    axes[0].set(xlabel="Step", ylabel="Total loss", title="Total Loss")

    axes[1].plot([r["step"] for r in train_h], [r["xct_loss"] for r in train_h], lw=0.6, label="xct")
    axes[1].plot([r["step"] for r in train_h], [r.get("mask_bce", 0) for r in train_h], lw=0.6, label="mask_bce")
    axes[1].set(xlabel="Step", title="Component Losses")
    axes[1].legend(fontsize=8)

    axes[2].plot([r["step"] for r in train_h], [r.get("kl", 0) for r in train_h], lw=0.6, color="red")
    axes[2].set(xlabel="Step", title="KL Divergence")

    plt.tight_layout()
    plt.show()
else:
    print("No training history to plot.")

### J.2) Visualize Reconstruction Slices

In [ ]:
model.eval()
with torch.no_grad():
    vis_batch = next(iter(val_loader))
    xct_in = vis_batch["xct"].to(device)
    mask_in = vis_batch["mask"].to(device)

    with torch.autocast(device_type=device.type, dtype=autocast_dtype):
        out = model(xct_in, mask_in)

    xct_pred = torch.sigmoid(out.xct_logits).float().cpu()
    mask_pred = (torch.sigmoid(out.mask_logits) > 0.5).float().cpu()

# Show first 4 samples, mid-slice
n_show = min(4, xct_pred.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(12, 3 * n_show))
if n_show == 1:
    axes = axes[None, :]

for i in range(n_show):
    mid = xct_pred.shape[2] // 2
    axes[i, 0].imshow(vis_batch["xct"][i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 0].set_title("XCT GT")
    axes[i, 1].imshow(xct_pred[i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("XCT Pred")
    axes[i, 2].imshow(vis_batch["mask"][i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("Mask GT")
    axes[i, 3].imshow(mask_pred[i, 0, mid].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 3].set_title("Mask Pred")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Reconstruction — Mid-Slice (Z)")
plt.tight_layout()
plt.show()

### J.3) Dice Score on Validation Batches

In [ ]:
model.eval()
all_dice = []
all_iou = []
n_eval_batches = 10

with torch.no_grad():
    for i, vb in enumerate(val_loader):
        if i >= n_eval_batches:
            break
        xct_in = vb["xct"].to(device)
        mask_in = vb["mask"].to(device)
        with torch.autocast(device_type=device.type, dtype=autocast_dtype):
            out = model(xct_in, mask_in)
        seg = segmentation_metrics(out.mask_logits.float(), mask_in.float())
        all_dice.append(seg["dice_all"])
        all_iou.append(seg["iou_all"])

print(f"Dice (all, {n_eval_batches} batches): {sum(all_dice)/len(all_dice):.4f}")
print(f"IoU  (all, {n_eval_batches} batches): {sum(all_iou)/len(all_iou):.4f}")

### J.4) Overfit Sanity Test (Optional)

Re-run training with `CFG["overfit_one_batch"] = True` above.  
If the model is correct, loss should drop to near-zero within 100–200 steps.